# Streaming data processing

1. Check if the Kafka server has any defined topics:
    - in the additional terminal window, enter the command:
```bash
cd ~ 
kafka/bin/kafka-topics.sh --list --bootstrap-server broker:9092
```

2. Create a topic called `groupXj1xx` - change X to the group number and for xx put the number of your python environment.
```bash
cd ~ 
kafka/bin/kafka-topics.sh --bootstrap-server broker:9092 --create --topic grupaXj1xx
```

3. Check again the topic's list. You should see your topic `grupaXj1xx`.

4. Run a new terminal window and console producer for the topic you created.
```bash
cd ~ 
kafka/bin/kafka-console-producer.sh --bootstrap-server broker:9092 --topic grupaXj1xx
```

We want to check if the broker server works. Run a new terminal and console consumer for the topic you created:
```bash
cd ~ 
kafka/bin/kafka-console-consumer.sh  --bootstrap-server broker:9092 --topic grupaXj1xx 
```
Put some text in the producer window and check consumer.  

Close producer terminal. 

Console Consumer will be a good place to check out your next program. 

## The code for streaming data

Complete the script so that it generates the following data: 

1. Create the `message` variable for one event (key: value): 
    - "time" : datetime.now() as a string
    - "id" : random choose from a list ["a", "b", "c", "d", "e"]
    - "value: random value from 0 to 100

In [1]:
%%file stream.py

import json
import random
import sys
from datetime import datetime
from time import sleep
from kafka import KafkaProducer


KAFKA_SERVER = "broker:9092"
TOPIC = ...
LAG = 2

def create_producer(server):
    return KafkaProducer(
        bootstrap_servers=[server],
        value_serializer=lambda x: json.dumps(x).encode("utf-8"),
        api_version=(3, 7, 0),
    )

if __name__ == "__main__":
    
    producer = create_producer(KAFKA_SERVER)
    try:
        while True:

        ### YOUR CODE
            message = {}
        ###    
            producer.send(TOPIC, value=message)
            sleep(LAG)
    except KeyboardInterrupt:
        producer.close()


Writing stream.py


2.  Run new terminal and `stream.py` script
```bash
python stream.py
```

Check console consumer window if you see new events.

## APACHE SPARK 

Prepare the script code to retrieve information from the Apache Kafka data stream.

In [2]:
%%file app.py

## LOAD SPARK SESSION object

SERVER = "broker:9092"
TOPIC = ...

if __name__ == "__main__":
    ## create spark variable
    #YOUR CODE HERE
    
    ##  
    spark.sparkContext.setLogLevel("WARN")
     
    raw = (
        spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", SERVER)
        .option("subscribe", TOPIC)
        .load()
    )

    query =  (
        raw.writeStream
        .outputMode("append")
        .format("console")
        .option("truncate", False)
        .start()
    )
    
    
    query.awaitTermination()
    query.stop()

Writing app.py


run your first spark code: 
```bash
spark-submit --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.1.1 app.py
```


Modify `app.py` file:

1. Add schema of event:
   
```python
    json_schema = StructType(
        [
            StructField("time", TimestampType()),
            StructField("id", StringType()),
            StructField("value", IntegerType()),
        ]
    )
```
You can use ddl schema string. 

Add stream data transformation and get `value` field from Kafka DataFrame. 

The `f` is `pyspark.sql.functions` library - import it.


```python
    parsed = raw.select(
        "timestamp", f.from_json(raw.value.cast("string"), json_schema).alias("json")
    ).select(
        f.col("timestamp").alias("proc_time"),
        f.col("json").getField("time").alias("event_time"),
        f.col("json").getField("id").alias("id"),
        f.col("json").getField("value").alias("value"),
    )
```
or with decoding function

```python
parsed = raw.select("timestamp", f.from_json(f.decode(f.col("value"), "utf-8"), schema).alias("values")
                   ).select("timestamp", "values.*")
```

In many tutorials You can find that solution 
```python
parsed = raw.selectExpr("cast(value as string) as value") 
```
but i don't know how to use it. It will give you just ordinary string not json struct.

Run a new code version.  

## Count number of events group by ID column

look at this error: 

`Append output mode not supported when there are streaming aggregations on streaming DataFrames/DataSets without watermark;`

you can use "complete" or "update" output mode.

In the simple case you can add trigger time.
```python
writeStream.trigger(processingTime='5 seconds')
```
## Window straming

For a tumblink window you can use `window` function with one time parameter.

```python
grouped = parsed.groupBy(f.window("timestamp", "5 seconds"),"id").count()
```

For sliding window with height 10 seconds and run new window after 5 seconds you can use: 

```python
grouped = parsed.groupBy(f.window("timestamp", "10 seconds", "5 seconds")).count()
```

Check "complete" and "update" output mode. 